# 레이트 청킹 (Late Chunking) — 법령 조문 단위

환경: chunk (Python 3.11.15 / transformers 4.46.3 / tokenizers 0.20.3 / sentence_transformers 3.4.1 / chonkie 1.3.1)

**일반 청킹**은 조문을 먼저 잘라서 따로따로 임베딩한다 → 각 조각이 앞뒤 문맥을 모른다.
**레이트 청킹**은 반대다:
1. 문서를 통째로 BGE-M3에 넣어 `last_hidden_state`(토큰별 벡터)를 뽑고
2. *그 다음* 조문 경계로 토큰 벡터를 잘라 평균낸다 → 각 조문 임베딩이 문서 전체 문맥을 머금는다

핵심은 **풀링 전 토큰 벡터**에 접근하는 것. OpenAI 임베딩 API는 이걸 안 열어줘서 레이트 청킹이 불가능하고,
BGE-M3는 오픈웨이트라 `transformers`로 직접 forward를 돌려 그 텐서를 손에 쥘 수 있다.

## 0. 임포트 & 설정

In [ ]:
import re, json, torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

MODEL       = "BAAI/bge-m3"
INPUT_PATH  = r"C:\Users\User\Desktop\민원처리법_hwpx변환.md"
OUTPUT_PATH = r"C:\Users\User\Desktop\민원처리법_late_chunks.json"

MAX_LEN = 8192          # BGE-M3 최대 컨텍스트. 초과 시 조문 단위 윈도우로 분할
device  = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: gpu


## 1. 조문 경계 파서

레이트 청킹은 "경계"가 필요하다. 법령은 조문이 자연 경계라 이걸 쓴다.
각 조문의 **문자 시작/끝 위치(start, end)** 를 잡아둔다 — 나중에 토큰 위치로 변환할 것.
(제N조의M 가지번호, 제목 없는 조, 부칙, 절 없는 장 모두 처리)

In [2]:
RE_TITLE   = re.compile(r'^#\s+(.+)$')
RE_META_SI = re.compile(r'\[시행\s*([\d.\s]+?)\]')
RE_META_NO = re.compile(r'\[법률\s*(제\d+호)')
RE_ABBR    = re.compile(r'약칭:\s*([^)]+)')
RE_CHAP    = re.compile(r'^##\s+(제\d+장.*)$')
RE_SEC     = re.compile(r'^###\s+(제\d+절.*)$')
RE_ADDENDA = re.compile(r'^#{1,3}\s*부칙')
RE_JO      = re.compile(r'^\*\*(제\d+조(?:의\d+)?)\(([^)]*)\)\*\*\s*(.*)$')
RE_JO_NT   = re.compile(r'^\*\*(제\d+조(?:의\d+)?)\*\*\s*(.*)$')

def parse_law(text):
    lines = text.splitlines(keepends=True)   # keepends: 문자 offset 정확히 유지
    # 메타
    meta = {"법령명": None, "약칭": None, "시행일": None, "법령번호": None}
    for ln in lines[:10]:
        m = RE_TITLE.match(ln)
        if m and meta["법령명"] is None:
            t = m.group(1); meta["법령명"] = re.sub(r'\s*\(약칭:.*\)', '', t).strip()
            a = RE_ABBR.search(t);  meta["약칭"] = a.group(1).strip() if a else None
        if meta["시행일"] is None:
            s = RE_META_SI.search(ln); meta["시행일"] = re.sub(r'\s+',' ',s.group(1)).strip() if s else meta["시행일"]
        if meta["법령번호"] is None:
            n = RE_META_NO.search(ln); meta["법령번호"] = n.group(1) if n else meta["법령번호"]

    chunks = []
    chapter = section = None; part = "본칙"
    buf = None; pos = 0   # pos: 현재 줄의 문자 시작 offset

    def flush(end):
        nonlocal buf
        if buf is None: return
        body = text[buf["start"]:end].strip()
        buf.update(content=body, start_index=buf["start"], end_index=end)
        del buf["start"]
        chunks.append(buf); buf = None

    for ln in lines:
        stripped = ln.rstrip("\n")
        if RE_ADDENDA.match(stripped):
            flush(pos); part = "부칙"; chapter = section = None
        elif RE_CHAP.match(stripped):
            flush(pos); chapter = RE_CHAP.match(stripped).group(1).strip(); section = None
        elif RE_SEC.match(stripped):
            flush(pos); section = RE_SEC.match(stripped).group(1).strip()
        else:
            mj = RE_JO.match(stripped) or RE_JO_NT.match(stripped)
            if mj:
                flush(pos)
                has_title = mj.re is RE_JO
                buf = {"chunk_id": len(chunks), **meta, "구분": part,
                       "장": chapter, "절": section,
                       "조번호": mj.group(1),
                       "조제목": mj.group(2).strip() if has_title else None,
                       "start": pos}
        pos += len(ln)
    flush(len(text))
    return meta, chunks

with open(INPUT_PATH, encoding="utf-8") as f:
    text = f.read()
meta, chunks = parse_law(text)
print("메타:", json.dumps(meta, ensure_ascii=False))
print(f"→ {len(chunks)}개 조문 경계 확보")
for c in chunks[:3]:
    print(f'  [{c["chunk_id"]}] {c["조번호"]} @ ({c["start_index"]},{c["end_index"]})  {c["content"][:30]}...')

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\User\\Desktop\\민원처리법_hwpx변환.md'

## 2. BGE-M3 로드 (transformers 직접 — 토큰 벡터 접근용)

`sentence_transformers`가 아니라 `AutoModel`로 로드해야 `last_hidden_state`를 꺼낼 수 있다. 이게 레이트 청킹의 전제.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModel.from_pretrained(MODEL).to(device).eval()
print("BGE-M3 로드 완료 · hidden dim:", model.config.hidden_size)

## 3. 레이트 청킹 핵심

- `encode_tokens`: 텍스트 구간을 forward → 토큰 벡터(`last_hidden_state`) + 각 토큰의 문자 offset
- `char_to_token_span`: 조문의 문자 범위를 토큰 인덱스 범위로 변환
- `late_chunk`: 문서를 8192 이하 윈도우(조문 단위로 패킹)로 나눠 forward하고, 조문별로 토큰 벡터를 평균

전체 문서가 8192 토큰 이하면 한 번의 forward로 끝난다(진짜 full-context).
초과하면 조문 경계로 윈도우를 쪼개되 **한 조문이 두 윈도우에 걸치지 않게** 패킹한다.

In [ ]:
@torch.no_grad()
def encode_tokens(segment_text):
    enc = tokenizer(segment_text, return_offsets_mapping=True, return_tensors="pt",
                    truncation=True, max_length=MAX_LEN, add_special_tokens=True)
    offsets = enc.pop("offset_mapping")[0].tolist()
    out = model(**{k: v.to(device) for k, v in enc.items()})
    return out.last_hidden_state[0], offsets     # [seq_len, dim], [(s,e)...]

def char_to_token_span(cs, ce, offsets):
    ts = te = None
    for i, (s, e) in enumerate(offsets):
        if e <= s:      continue        # 스페셜 토큰 (0,0)
        if e <= cs:     continue        # 조문 시작 전
        if s >= ce:     break           # 조문 끝 이후
        if ts is None:  ts = i
        te = i + 1
    return ts, te

def late_chunk(text, chunks):
    # 조문의 대략 토큰수 = 문자수/2 로 추정해 윈도우 패킹 (한글 기준 보수적)
    est = lambda c: max(1, (c["end_index"] - c["start_index"]) // 2)
    budget = MAX_LEN - 16
    late = [None] * len(chunks)
    i = 0
    while i < len(chunks):
        j, tok = i, 0
        while j < len(chunks) and tok + est(chunks[j]) <= budget:
            tok += est(chunks[j]); j += 1
        j = max(j, i + 1)                         # 한 조문이 budget보다 커도 최소 1개
        w_start = chunks[i]["start_index"]
        w_end   = chunks[j-1]["end_index"]
        tok_emb, offsets = encode_tokens(text[w_start:w_end])
        for k in range(i, j):                     # 이 윈도우 안 조문들 pooling
            cs = chunks[k]["start_index"] - w_start
            ce = chunks[k]["end_index"]   - w_start
            ts, te = char_to_token_span(cs, ce, offsets)
            vec = tok_emb.mean(0) if ts is None else tok_emb[ts:te].mean(0)
            late[k] = F.normalize(vec, dim=0).cpu()
        i = j
    return torch.stack(late)

late_emb = late_chunk(text, chunks)
print("레이트 임베딩:", tuple(late_emb.shape))   # (조문수, 1024)

## 4. (비교) naive 청킹 — 조문을 따로따로 임베딩

같은 조문을 문맥 없이 독립 임베딩. 레이트 방식과 코사인으로 비교해 '문맥이 임베딩을 얼마나 바꾸는지' 본다.

In [ ]:
@torch.no_grad()
def naive_chunk(chunks):
    embs = []
    for c in chunks:
        enc = tokenizer(c["content"], return_tensors="pt", truncation=True, max_length=MAX_LEN)
        out = model(**{k: v.to(device) for k, v in enc.items()})
        embs.append(F.normalize(out.last_hidden_state[0].mean(0), dim=0).cpu())
    return torch.stack(embs)

naive_emb = naive_chunk(chunks)

cos = (late_emb * naive_emb).sum(1)          # 둘 다 정규화됨 → 내적 = 코사인
order = cos.argsort()                          # 낮을수록 문맥으로 많이 바뀐 조문
print("=== 문맥 영향 큰 조문 TOP 5 (late vs naive 코사인 낮은 순) ===")
for idx in order[:5].tolist():
    c = chunks[idx]
    print(f'  cos={cos[idx]:.3f}  {c["조번호"]}({c["조제목"]})  {c["content"][:34].strip()}...')
print(f"\n전체 평균 코사인: {cos.mean():.3f}  (1에 가까울수록 문맥 영향 적음)")

## 5. 저장

조문 메타 + 레이트 임베딩을 JSONL로. 그대로 벡터DB(Qdrant 등) upsert에 흘린다.

In [ ]:
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for c, v in zip(chunks, late_emb):
        rec = {**{k: c[k] for k in ["chunk_id","법령명","구분","장","절","조번호","조제목","content"]},
               "embedding": v.tolist()}
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
print(f"저장 완료: {OUTPUT_PATH}  ({len(chunks)}개 · dim {late_emb.shape[1]})")